In [0]:
%python
spark.sql("""
    SELECT MIN(OrderDate) AS min_date, MAX(OrderDate) AS max_date
    FROM workspace.silver.sales_order_header
""").show()

In [0]:
%python
from pyspark.sql.functions import (
    explode, sequence, to_date, col, year, quarter, month,
    dayofmonth, dayofweek, date_format, weekofyear
)

date_df = spark.sql("""
    SELECT explode(sequence(to_date('2011-01-01'), to_date('2014-12-31'), interval 1 day)) AS full_date
""")

dim_date = (date_df
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("day_of_month", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn("is_weekend", col("day_of_week").isin(1, 7))
)

print(f"Total dates: {dim_date.count()}")
dim_date.show(5)

dim_date.write.mode("overwrite").saveAsTable("workspace.gold.dim_date")

In [0]:
%python
from pyspark.sql.functions import col

product_df = spark.table("workspace.silver.product")

dim_product = (product_df.select(
    col("ProductID").alias("product_id"),
    col("Name").alias("product_name"),
    col("ProductNumber").alias("product_number"),
    col("Color").alias("color"),
    col("Size").alias("size"),
    col("Weight").alias("weight"),
    col("StandardCost").alias("standard_cost"),
    col("ListPrice").alias("list_price"),
    col("ProductLine").alias("product_line"),
    col("Class").alias("class"),
    col("Style").alias("style"),
    col("MakeFlag").alias("is_make"),
    col("FinishedGoodsFlag").alias("is_finished_good"),
    col("SellStartDate").alias("sell_start_date"),
    col("SellEndDate").alias("sell_end_date"),
))

print(f"dim_product rows: {dim_product.count()}")
dim_product.show(5)

dim_product.write.mode("overwrite").saveAsTable("workspace.gold.dim_product")

In [0]:
%python
from pyspark.sql.functions import col

customer_df = spark.table("workspace.silver.customer")
territory_df = spark.table("workspace.silver.sales_territory")

dim_customer = (customer_df
    .join(territory_df.select(col("TerritoryID"), col("Name").alias("territory_name"),
                               col("CountryRegionCode"), col("Group").alias("territory_group")),
          "TerritoryID", "left")
    .select(
        col("CustomerID").alias("customer_id"),
        col("PersonID").alias("person_id"),
        col("StoreID").alias("store_id"),
        col("AccountNumber").alias("account_number"),
        col("TerritoryID").alias("territory_id"),
        col("territory_name"),
        col("CountryRegionCode").alias("country_region_code"),
        col("territory_group"),
    )
)

print(f"dim_customer rows: {dim_customer.count()}")
dim_customer.show(5)

dim_customer.write.mode("overwrite").saveAsTable("workspace.gold.dim_customer")

In [0]:
%python
from pyspark.sql.functions import col, to_date, date_format

header_df = spark.table("workspace.silver.sales_order_header")
detail_df = spark.table("workspace.silver.sales_order_detail")

fact_sales = (detail_df
    .join(header_df, "SalesOrderID", "inner")
    .select(
        col("SalesOrderDetailID").alias("sales_order_detail_id"),
        col("SalesOrderID").alias("sales_order_id"),
        date_format(col("OrderDate"), "yyyyMMdd").cast("int").alias("order_date_key"),
        col("CustomerID").alias("customer_id"),
        col("ProductID").alias("product_id"),
        col("TerritoryID").alias("territory_id"),
        col("OrderQty").alias("order_qty"),
        col("UnitPrice").alias("unit_price"),
        col("UnitPriceDiscount").alias("unit_price_discount"),
        col("LineTotal").alias("line_total"),
        col("SubTotal").alias("order_subtotal"),
        col("TaxAmt").alias("order_tax_amt"),
        col("Freight").alias("order_freight"),
        col("TotalDue").alias("order_total_due"),
        col("OnlineOrderFlag").alias("is_online_order"),
        col("Status").alias("order_status"),
    )
)

print(f"fact_sales rows: {fact_sales.count()}")

# Sanity check: fact grain should match sales_order_detail row count exactly
print(f"sales_order_detail rows: {detail_df.count()}")

fact_sales.write.mode("overwrite").saveAsTable("workspace.gold.fact_sales")